# HE: Homomorphic Encryption for Secure Inference

**Library:** [TenSEAL](https://github.com/OpenMined/TenSEAL) 0.3+ (OpenMined, Apache-2.0), built on [Microsoft SEAL](https://github.com/microsoft/SEAL)  
**Install:** `pip install tenseal`  
**Stage:** Inference (compute on encrypted patient data without decrypting)

---

## What HE Does

Homomorphic Encryption allows computation on ciphertext. A hospital encrypts patient data, sends it to a cloud model, and gets encrypted predictions back — the cloud **never sees the plaintext data**.

```
Hospital:  encrypt(patient_data)  -->  Cloud: model(encrypted_data)  -->  Hospital: decrypt(prediction)
                                       Cloud never sees raw data!
```

Supported schemes:
- **CKKS:** Approximate arithmetic on real numbers (ML inference)
- **BFV:** Exact integer arithmetic (counting, aggregation)

In [ ]:
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)
print(f'Working directory: {os.getcwd()}')

## Step 1: Create Encryption Context

In [ ]:
from fl_pets.he import create_context, encrypt, decrypt
import tenseal

print(f'TenSEAL {tenseal.__version__} (Microsoft SEAL)')

ctx = create_context(scheme='ckks', poly_mod_degree=8192)
print(f'CKKS context: poly_mod_degree=8192, scale=2^40')

## Step 2: Encrypted Arithmetic

In [ ]:
# Encrypt two vectors
a = [3.14, -1.5, 99.9, 0.001]
b = [2.0, 3.0, 5.0, 10.0]
ea, eb = encrypt(ctx, a), encrypt(ctx, b)

# Homomorphic operations
dec_add = decrypt(ea + eb)
dec_mul = decrypt(ea * eb)

print(f'a = {a}')
print(f'b = {b}')
print(f'\nEncrypted a + b: [{" ".join(f"{x:.4f}" for x in dec_add)}]')
print(f'Expected  a + b: [{" ".join(f"{x+y:.4f}" for x,y in zip(a,b))}]')
print(f'\nEncrypted a * b: [{" ".join(f"{x:.4f}" for x in dec_mul)}]')
print(f'Expected  a * b: [{" ".join(f"{x*y:.4f}" for x,y in zip(a,b))}]')
print(f'\nAdd error: {max(abs(d-(x+y)) for d,x,y in zip(dec_add,a,b)):.2e}')
print(f'Mul error: {max(abs(d-(x*y)) for d,x,y in zip(dec_mul,a,b)):.2e}')

## Step 3: Practical Limitations

HE is **not** a general-purpose replacement for plaintext computation:

| Operation | CKKS support | Notes |
|-----------|-------------|-------|
| Addition | Yes | Fast, unlimited depth |
| Multiplication | Yes | Limited depth (each mul adds noise) |
| Comparison (>, <) | No | Use MPC instead |
| Division | No | Approximate via multiplication |
| Non-linear (ReLU, sigmoid) | Polynomial approx only | Low accuracy for deep networks |

**Best for:** Small models (MLP, logistic regression, linear models).  
**Not practical for:** DenseNet-121, ResNet-50, transformers — use MPC (CrypTen) instead.

## References

- [TenSEAL](https://github.com/OpenMined/TenSEAL) — OpenMined, Apache-2.0
- [Microsoft SEAL](https://github.com/microsoft/SEAL) — MIT License
- Cheon et al. (2017), *Homomorphic Encryption for Arithmetic of Approximate Numbers* (CKKS)

## Next Steps

- [MPC: Secure Inference](mpc-secure-inference.ipynb) — for larger models (DenseNet, BERT)
- [PET Reference](../reference/PET_Reference.md) — full technical comparison